### **set up & Import Library**

In [ ]:
import os
from google.colab import files
import google.auth
import numpy as np
import pandas as pd
from google.cloud import bigquery

# 1. Memunculkan tombol upload agar kamu bisa memilih file JSON dari laptop/komputer
print("Silakan upload file JSON Service Account:")
uploaded = files.upload()

# Mengambil nama file yang baru saja di-upload dan membuat path-nya
file_name = list(uploaded.keys())[0]
file_path = f"/content/{file_name}"

# 2. Daftarkan file JSON Service Account ke environment variable
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = file_path
print(f"\nBerhasil! Kredensial telah diset ke: {file_path}\n")

# 3. Tambahkan izin (scopes) agar Service Account bisa membaca BigQuery dan Google Drive
scopes = (
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/drive"
)

# 4. Buat kredensial otorisasi beserta scopes-nya
credentials, project_id = google.auth.default(scopes=scopes)

# 5. Inisialisasi klien BigQuery menggunakan kredensial yang sudah memiliki akses Drive
client = bigquery.Client(credentials=credentials, project=project_id)

In [ ]:
query = """
with stockawal as
(
select
artikel,sub_kategori,
sum(stock) stock,
min(s.tanggal) tanggal,
date_sub(min(s.tanggal),interval 90 day) tanggal_awal,
-- date_add(min(s.tanggal),interval 30 day) tanggal_akhir,
-- date_add(min(s.tanggal),interval 60 day) tanggal_akhir2
from `klamby-469003.sales.stockawal` s
left join `klamby-469003.sales.master_sku_gabungan` ms on s.sku = ms.sku
where keterangan != 'Restock' and ms.artikel is not null and ms.artikel not like '%Minor%' and ms.artikel not like '%Major%'
group by 1,2
),
sold as
(
select date_trunc(created_at, day) tanggal, ms.artikel, sum(quantity) quantity
from (select * from `klamby-469003.sales.daily` union all select * from `klamby-469003.sales.monthly`) n
left join `klamby-469003.sales.master_sku_gabungan` ms on n.sku = ms.sku
where ms.artikel is not null and ms.artikel not like '%Minor%' and ms.artikel not like '%Major%'
and (status_id = 101 or status_id = 401)
group by 1,2
),
product_attribute as
(
SELECT
*,
row_number() over(partition by artikel) rn
FROM `klamby-469003.sales.product_attribute`
qualify rn = 1
)
select
s.artikel, s.tanggal,s.sub_kategori, stock,
upper(p.cutting) cutting, upper(p.design_density) design_density, upper(p.design_border) design_border,upper(p.siluet) siluet,
upper(p.sleeve) sleeve,upper(p.collar) collar,upper(p.length_hem) length_hem,upper(p.color_theme) color_theme,upper(p.closure) closure,upper(p.embellishment) embellishment,
sum(so.quantity) quantity,
sum(case when so.tanggal >= s.tanggal_awal and so.tanggal <= date_add(s.tanggal, interval 30 day) then so.quantity else 0 end) as quantity30,
sum(case when so.tanggal >= s.tanggal_awal and so.tanggal <= date_add(s.tanggal, interval 60 day) then so.quantity else 0 end) as quantity60,
sum(case when so.tanggal >= s.tanggal_awal and so.tanggal <= date_add(s.tanggal, interval 90 day) then so.quantity else 0 end) as quantity90
from stockawal s
left join sold so on s.artikel = so.artikel and so.tanggal >= tanggal_awal --and so.tanggal <= tanggal_akhir
--left join sold so2 on s.artikel = so2.artikel and so.tanggal >= tanggal_awal and so2.tanggal <= tanggal_akhir2
--where s.artikel = 'Elaya Sock'
left join product_attribute p on UPPER(s.artikel) = p.artikel
where p.artikel is not null AND DATE(s.tanggal) <= DATE_SUB(CURRENT_DATE('Asia/Bangkok'), INTERVAL 90 DAY)
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14

"""
df = client.query(query).to_dataframe()
df.head()


### **cleaning & preparation**

In [ ]:
df.info()

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Tanggal launching mengikuti tanggal stok awal, sesuai query notebook.
as_of = pd.Timestamp(datetime.now(ZoneInfo('Asia/Bangkok')).date())
launch_dates = pd.to_datetime(df['tanggal'], errors='coerce').dt.normalize()
df_eligible = df.loc[launch_dates.notna() & (launch_dates <= as_of - pd.Timedelta(days=90))].copy()
df_eligible['tanggal'] = pd.to_datetime(df_eligible['tanggal']).dt.normalize()
df_eligible['stock'] = pd.to_numeric(df_eligible['stock'], errors='coerce')
df_eligible = df_eligible.loc[df_eligible['artikel'].notna() & np.isfinite(df_eligible['stock']) & (df_eligible['stock'] > 0)].reset_index(drop=True)
if df_eligible['artikel'].duplicated().any():
    raise ValueError('Artikel duplikat; periksa sumber data.')
if len(df_eligible) < 20:
    raise ValueError('Minimal 20 artikel valid berusia >=90 hari diperlukan.')
df_articles = df_eligible[['artikel', 'tanggal']].copy()
df_reg = df_eligible.copy()
df_reg['bulan'] = df_reg['tanggal'].dt.month.astype(str)
df_reg['tahun'] = df_reg['tanggal'].dt.year
cat_features = ['sub_kategori', 'cutting', 'design_density', 'design_border', 'siluet', 'sleeve', 'collar', 'length_hem', 'color_theme', 'bulan']
df_reg[cat_features] = df_reg[cat_features].astype(object).where(df_reg[cat_features].notna(), 0).astype(str)
for days in (30, 60, 90):
    quantity = pd.to_numeric(df_reg[f'quantity{days}'], errors='raise').fillna(0)
    if not np.isfinite(quantity).all() or (quantity < 0).any():
        raise ValueError('Quantity harus finite dan non-negatif.')
    df_reg[f'str{days}'] = (quantity / df_reg['stock']).clip(upper=1)
df_reg = df_reg.drop(columns=['tanggal','artikel','closure','embellishment','quantity','quantity30','quantity60','quantity90'])


In [ ]:
df_reg.info()

In [ ]:
df_reg['sub_kategori'].sort_values().unique()

In [ ]:
df_reg['cutting'].astype(str).sort_values().unique()

In [ ]:
df_reg['design_density'].astype(str).sort_values().unique()

In [ ]:
df_reg['design_border'].astype(str).sort_values().unique()

In [ ]:
df_reg['siluet'].astype(str).sort_values().unique()

In [ ]:
df_reg['sleeve'].astype(str).sort_values().unique()

In [ ]:
df_reg['collar'].astype(str).sort_values().unique()

In [ ]:
df_reg['length_hem'].astype(str).sort_values().unique()

In [ ]:
df_reg['color_theme'].astype(str).sort_values().unique()

In [ ]:
df_reg['bulan'].astype(int).sort_values().unique()

### **ml setup**

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
categorical_columns = df_reg.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(df_reg[categorical_columns].astype(str))

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns)
)

final_df = pd.concat(
    [df_reg.drop(columns=categorical_columns), encoded_df],
    axis=1
)

In [ ]:
final_df

In [ ]:
def print_metrics(y_true, preds, model_name):
  print('Model: '+ str(model_name))
  print('Mean Squared Error: ', format(round(mean_squared_error(y_true, preds),3)))
  print('Mean Absolute Error: ', format(round(mean_absolute_error(y_true, preds),3)))
  print('R2 Score: ', format(round(r2_score(y_true, preds),3)))

In [ ]:
X = final_df.drop(columns=['str30','str60','str90','stock','tahun'])
y = final_df[['str30','str60','str90','stock']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=43)

In [ ]:
tree_mod = DecisionTreeRegressor()
rf_mod = RandomForestRegressor()
ada_mod = MultiOutputRegressor(AdaBoostRegressor())
reg_mod = LinearRegression()
La = Lasso(alpha=0.5)
ri = Ridge(alpha=0.9)

In [ ]:
# 1. Tentukan grid parameter yang ingin diuji
param_grid_rf = {
    'n_estimators': [50, 100, 200],      # Jumlah pohon di hutan
    'max_depth': [None, 10, 20, 30],     # Kedalaman maksimal pohon
    'min_samples_split': [2, 5, 10],     # Minimal sampel untuk membelah node
    'min_samples_leaf': [1, 2, 4]        # Minimal sampel di ujung daun (leaf)
}

# 2. Inisialisasi GridSearchCV
# cv=5 berarti menggunakan 5-fold cross validation
# n_jobs=-1 agar proses berjalan lebih cepat dengan memakai seluruh core CPU
grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=43),
    param_grid=param_grid_rf,
    cv=5,
    scoring='r2', # Bisa diganti 'neg_mean_squared_error' jika fokus ke MSE
    n_jobs=-1,
    verbose=2
)

# 3. Latih Grid Search menggunakan data training
print("Mulai proses tuning Random Forest...")
grid_rf.fit(X_train, y_train) #

# 4. Tampilkan parameter terbaik hasil tuning
print("Parameter Terbaik RF:", grid_rf.best_params_)

# 5. Evaluasi model terbaik menggunakan data testing
best_rf_mod = grid_rf.best_estimator_
preds_rf_tuned = best_rf_mod.predict(X_test) #
print_metrics(y_test, preds_rf_tuned, "Tuned Random Forest") #

In [ ]:
tree_mod.fit(X_train, y_train)
rf_mod.fit(X_train, y_train)
ada_mod.fit(X_train, y_train)
reg_mod.fit(X_train, y_train)
La.fit(X_train, y_train)
ri.fit(X_train, y_train)

In [ ]:
preds_tree = tree_mod.predict(X_test)
preds_rf = rf_mod.predict(X_test)
preds_ada = ada_mod.predict(X_test)
preds_reg = reg_mod.predict(X_test)
preds_lasso = La.predict(X_test)
preds_rigde = ri.predict(X_test)

In [ ]:
print_metrics(y_test,preds_tree,tree_mod)

In [ ]:
print_metrics(y_test,preds_rf,rf_mod)

In [ ]:
print_metrics(y_test,preds_ada,ada_mod)

In [ ]:
print_metrics(y_test,preds_reg,reg_mod)

In [ ]:
print_metrics(y_test,preds_lasso,La)

In [ ]:
print_metrics(y_test,preds_rigde,ri)

### **coba implementasi**

In [ ]:
import joblib

# 1. Simpan Model Terbaik dan Encoder
joblib.dump(best_rf_mod, 'model_rf_terbaik.pkl')
joblib.dump(encoder, 'encoder_kategori.pkl')
joblib.dump(categorical_columns, 'cat_cols.pkl') # Simpan nama kolom kategorikal
print("Model dan Encoder berhasil disimpan!")

In [ ]:
data_baru = pd.DataFrame({
    'sub_kategori': ['SHIRT'],
    'cutting': ['REGULAR'],
    'design_density': ['HEAVY'],
    'design_border': [0],
    'siluet': ['A-LINE'],
    'sleeve': ['LONG SLEEVE'],
    'collar': ['SHANGHAI COLLAR'],
    'length_hem': ['MEDIUM'],
    'color_theme': ['DUAL COLOR'],
    'bulan': ['10'],  # Ingat, bulan di kode Anda diubah jadi string
    ##'tahun': [2023]   # Numerik
})

In [ ]:
encoded_data_baru = encoder.transform(data_baru[categorical_columns].astype(str))
encoded_frame = pd.DataFrame(encoded_data_baru, columns=encoder.get_feature_names_out(categorical_columns))[X.columns]
predict = best_rf_mod.predict(encoded_frame)
print(f"Prediksi STR 30 Hari : {predict[0][0]:.2%}")
print(f"Prediksi STR 60 Hari : {predict[0][1]:.2%}")
print(f"Prediksi STR 90 Hari : {predict[0][2]:.2%}")
print(f"Estimasi stok awal historis : {predict[0][3]:,.0f} pcs")


### **coba deploy**
  dibawah ini hasil coding dari codex semua

In [ ]:
# Jalankan pada notebook revisi setelah grid_rf.fit selesai.
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
from zoneinfo import ZoneInfo
from uuid import uuid4
import json
import platform
import shutil
import joblib
import numpy as np
import pandas as pd
from google.colab import files

if 'df_articles' not in globals():
    raise ValueError('Jalankan ulang cell cleaning notebook revisi untuk menyimpan artikel dan tanggal launching.')
cutoff = pd.Timestamp(datetime.now(ZoneInfo('Asia/Bangkok')).date()) - pd.Timedelta(days=90)
if not (pd.to_datetime(df_articles['tanggal']) <= cutoff).all():
    raise ValueError('Data masih memuat produk <90 hari. Jalankan ulang cleaning dan training.')
if df_articles['artikel'].duplicated().any():
    raise ValueError('Artikel duplikat. Periksa data sumber sebelum training.')
cat_cols = list(categorical_columns)
input_df = df_reg[cat_cols].astype(object).where(df_reg[cat_cols].notna(), 0).astype(str)
dropdown_options = {c: input_df[c].unique().tolist() for c in cat_cols}
dropdown_options['bulan'] = sorted(dropdown_options['bulan'], key=int)
sample_values = encoder.transform(input_df.loc[X_train.index[:1]])
sample_x = pd.DataFrame(sample_values, columns=encoder.get_feature_names_out(cat_cols))[X.columns]
np.testing.assert_allclose(best_rf_mod.predict(sample_x), best_rf_mod.predict(X_train.iloc[[0]]))
references = df_articles.loc[X_train.index, ['artikel', 'tanggal']].copy()
references[cat_cols] = input_df.loc[X_train.index]
now = datetime.now(ZoneInfo('Asia/Bangkok'))
metadata = {
    'version': now.strftime('%Y%m%dT%H%M%S') + '-' + uuid4().hex[:8],
    'trained_at': now.isoformat(), 'as_of': str(now.date()), 'minimum_age_days': 90,
    'eligible_rows': len(df_reg), 'training_rows': len(X_train), 'test_rows': len(X_test),
    'best_params': grid_rf.best_params_,
    'library_versions': {p: version(p) for p in ['scikit-learn', 'numpy', 'pandas', 'scipy', 'joblib']},
}
folder = Path('/content/str_model_export')
folder.mkdir(exist_ok=True)
joblib.dump({
    'schema_version': 2, 'model_name': 'best_rf_mod', 'model': best_rf_mod,
    'encoder': encoder, 'categorical_columns': cat_cols, 'feature_columns': list(X.columns),
    'target_columns': list(y.columns), 'dropdown_options': dropdown_options,
    'training_articles': references.reset_index(drop=True), 'metadata': metadata,
}, folder / 'model_bundle.joblib', compress=3)
requirements = ['streamlit>=1.40,<2', 'google-cloud-storage>=3,<4']
requirements += [f'{p}=={v}' for p, v in metadata['library_versions'].items()]
(folder / 'requirements.txt').write_text('\n'.join(requirements) + '\n')
(folder / 'python-version.txt').write_text(platform.python_version() + '\n')
(folder / 'metadata.json').write_text(json.dumps(metadata, indent=2))
files.download(shutil.make_archive('/content/str_model_export', 'zip', folder))
